In [ ]:
# ============================================================
# KAGGLE RUNTIME - 08: MASTER PIPELINE (ORQUESTRADOR COMPLETO)
# ============================================================
# Este notebook roda no KAGGLE NOTEBOOK (GPU)
# Pipeline completo de inicialização do ambiente de execução
# 1. Preparar ambiente
# 2. Detectar GPU
# 3. Obter/atualizar código do GitHub
# 4. Preparar ComfyUI + custom nodes
# 5. Sincronizar modelos do Kaggle Dataset → SSD local
# 6. Health check da API
# 7. Iniciar ComfyUI
# 8. Informar que ambiente está pronto
# ============================================================

import sys
import subprocess
import time
from pathlib import Path

# Adicionar scripts ao path
sys.path.insert(0, "/kaggle/working/scripts")

print("=" * 70)
print("MASTER PIPELINE - KAGGLE NOTEBOOK RUNTIME")
print("=" * 70)

# ============================================================
# CONFIGURAÇÕES CENTRAIS
# ============================================================
CONFIG = {
    # Dataset
    "dataset": "automamermaid/comfydocs",
    
    # Diretórios
    "comfyui_dir": Path("/kaggle/working/ComfyUI"),
    "models_dir": Path("/kaggle/working/ComfyUI/models"),
    "scripts_dir": Path("/kaggle/working/scripts"),
    
    # GitHub
    "repo_url": "https://github.com/comfyanonymous/ComfyUI.git",
    
    # Custom nodes
    "custom_nodes": [
        "ltdrdata/ComfyUI-Manager",
        "cubiq/ComfyUI_essentials",
        "Kosinkadink/ComfyUI-VideoHelperSuite",
    ],
    
    # Categorias de modelos para sincronizar
    "model_categories": [
        "checkpoints",
        "diffusion_models",
        "loras",
        "vae",
        "text_encoders",
        "clip",
        "controlnet",
        "upscale_models",
        "video_models",
        "embeddings",
    ],
    
    # ComfyUI server
    "host": "0.0.0.0",
    "port": 8188,
    
    # Opções
    "force_sync": False,
    "auto_start": True,
    "health_check_timeout": 120,
}

def run_step(name, func, *args, **kwargs):
    """Executa um passo com logging padronizado"""
    print(f"\n{'='*70}")
    print(f"PASSO: {name}")
    print(f"{'='*70}")
    try:
        result = func(*args, **kwargs)
        print(f"✅ {name} - SUCESSO")
        return result
    except Exception as e:
        print(f"❌ {name} - FALHA: {e}")
        import traceback
        traceback.print_exc()
        raise

# ============================================================
# PASSO 1: Preparar ambiente e detectar GPU
# ============================================================
def step_prepare_env():
    from gpu_detect import detect_gpu, print_gpu_summary
    
    # Criar diretórios base
    for d in [CONFIG["comfyui_dir"], CONFIG["models_dir"], CONFIG["scripts_dir"]]:
        d.mkdir(parents=True, exist_ok=True)
    
    # Detectar GPU
    gpu_info = detect_gpu()
    print_gpu_summary(gpu_info)
    
    if not gpu_info["has_gpu"]:
        print("⚠️  AVISO: GPU não detectada! Performance será limitada.")
    
    return gpu_info

gpu_info = run_step("PREPARAR AMBIENTE + DETECTAR GPU", step_prepare_env)

# ============================================================
# PASSO 2: Obter/atualizar scripts compartilhados do GitHub
# ============================================================
def step_sync_scripts():
    import os
    scripts_repo = os.environ.get("SCRIPTS_GITHUB_REPO", "")
    if not scripts_repo:
        print("⚠️  SCRIPTS_GITHUB_REPO não configurado (env var). Pulando sync de scripts.")
        print("   Configure no Kaggle Secrets: SCRIPTS_GITHUB_REPO=https://github.com/user/repo.git")
        return
    scripts_dir = CONFIG["scripts_dir"]
    
    if (scripts_dir / ".git").exists():
        print("Atualizando scripts...")
        result = subprocess.run(
            ["git", "-C", str(scripts_dir), "pull"],
            capture_output=True, text=True
        )
    else:
        print("Clonando scripts...")
        result = subprocess.run(
            ["git", "clone", scripts_repo, str(scripts_dir)],
            capture_output=True, text=True
        )
    
    if result.returncode != 0:
        print(f"⚠️  Git falhou (pode ser normal se repo não existir): {result.stderr}")
        print("   Usando scripts locais se disponíveis")
    else:
        print("✅ Scripts sincronizados")

run_step("SINCRONIZAR SCRIPTS DO GITHUB", step_sync_scripts)

# ============================================================
# PASSO 3: Setup ComfyUI (clone/pull + deps + custom nodes)
# ============================================================
def step_setup_comfyui():
    from comfyui_setup import setup_comfyui
    
    return setup_comfyui(
        comfyui_dir=CONFIG["comfyui_dir"],
        repo_url=CONFIG["repo_url"],
        custom_nodes=CONFIG["custom_nodes"],
        models_dir=CONFIG["models_dir"],
    )

run_step("SETUP COMFYUI + CUSTOM NODES", step_setup_comfyui)

# ============================================================
# PASSO 4: Sincronizar modelos do Kaggle Dataset → SSD
# ============================================================
def step_sync_models():
    from kaggle_sync import sync_dataset_to_local
    
    stats = sync_dataset_to_local(
        dataset=CONFIG["dataset"],
        target_dir=CONFIG["models_dir"],
        categories=CONFIG["model_categories"],
        model_names=None,
        force=CONFIG["force_sync"],
    )
    
    print(f"Modelos sincronizados: {stats['synced']}")
    print(f"Modelos pulados: {stats['skipped']}")
    
    # Verificar modelo principal
    main_model = CONFIG["models_dir"] / "checkpoints" / "lustifyNSFWCheckpoint_v10Krea2.safetensors"
    if main_model.exists():
        size_gb = main_model.stat().st_size / (1024**3)
        print(f"✅ Modelo principal presente: {size_gb:.2f} GB")
    else:
        print(f"⚠️  Modelo principal NÃO encontrado: {main_model}")
    
    return stats

sync_stats = run_step("SYNC MODELOS KAGGLE DATASET → SSD", step_sync_models)

# ============================================================
# PASSO 5: Verificar instalação (health check pré-inicialização)
# ============================================================
def step_verify_install():
    comfyui_dir = CONFIG["comfyui_dir"]
    
    # Verificar arquivos essenciais
    essential = ["main.py", "requirements.txt", "comfy"]
    for f in essential:
        if not (comfyui_dir / f).exists():
            raise FileNotFoundError(f"Arquivo essencial não encontrado: {f}")
    
    # Verificar custom nodes
    custom_nodes_dir = comfyui_dir / "custom_nodes"
    if custom_nodes_dir.exists():
        nodes = list(custom_nodes_dir.iterdir())
        print(f"Custom nodes instalados: {len(nodes)}")
        for n in nodes:
            print(f"  - {n.name}")
    
    # Verificar modelos
    model_count = 0
    for cat in CONFIG["model_categories"]:
        cat_dir = CONFIG["models_dir"] / cat
        if cat_dir.exists():
            files = list(cat_dir.iterdir())
            model_count += len(files)
    print(f"Total de modelos no SSD: {model_count}")
    
    print("✅ Verificação de instalação OK")

run_step("VERIFICAR INSTALAÇÃO", step_verify_install)

# ============================================================
# PASSO 6: Iniciar ComfyUI
# ============================================================
def step_start_comfyui():
    from comfyui_setup import start_comfyui, health_check
    
    proc = start_comfyui(
        comfyui_dir=CONFIG["comfyui_dir"],
        host=CONFIG["host"],
        port=CONFIG["port"],
    )
    
    # Aguardar inicialização
    print(f"Aguardando ComfyUI inicializar (timeout: {CONFIG['health_check_timeout']}s)...")
    time.sleep(5)
    
    if health_check(CONFIG["host"], CONFIG["port"], CONFIG["health_check_timeout"]):
        print(f"\n🎉 COMFYUI RODANDO E RESPONDENDO!")
        print(f"   Acesse: http://{CONFIG['host']}:{CONFIG['port']}")
        print(f"   Ou use a URL do Kaggle Notebook (porta {CONFIG['port']})")
    else:
        print(f"\n⚠️  ComfyUI iniciou mas health check falhou")
        print(f"   Verifique logs em: {CONFIG['comfyui_dir']}")
    
    return proc

if CONFIG["auto_start"]:
    comfyui_proc = run_step("INICIAR COMFYUI", step_start_comfyui)
else:
    print("\n⏭️  Auto-start desabilitado. Inicie manualmente:")
    print(f"   cd {CONFIG['comfyui_dir']} && python main.py --listen {CONFIG['host']} --port {CONFIG['port']}")

# ============================================================
# RESUMO FINAL
# ============================================================
print("\n" + "=" * 70)
print("🎉 MASTER PIPELINE CONCLUÍDO - AMBIENTE PRONTO")
print("=" * 70)
print(f"Dataset: {CONFIG['dataset']}")
print(f"ComfyUI: {CONFIG['comfyui_dir']}")
print(f"Modelos: {CONFIG['models_dir']}")
print(f"GPU: {gpu_info['gpu_name'] if gpu_info['has_gpu'] else 'CPU only'}")
if CONFIG["auto_start"]:
    print(f"ComfyUI: http://{CONFIG['host']}:{CONFIG['port']}")
print("\nPara sync diário de modelos, execute: 07_sync_robust.ipynb")
print("Para adicionar modelos, use colab_transfer/02_download.ipynb + 03_upload_kaggle.ipynb")
print("=" * 70)